- 정상(N) 사기(P)
- Credit Card Fraud Detection
    - 99.89% 정상거래 0.17% 사기거래
- 중요한 평가지표 : Recall
    - 사기거래를 정상거래로 잘못 분류 (FN) - 고객 피해 및 장기적으로는 비즈니스의 문제
    - 정상거래를 사기거래로 잘못 분류 (FP) - 고객 달래기..
- 사기거래를 놓치면 안되는 경우 - 재현율

In [30]:

from sklearn.datasets import fetch_openml
from sklearn.metrics import confusion_matrix,precision_score,recall_score,f1_score,roc_curve,roc_auc_score
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from torch.optim import Adam

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

In [31]:
data = fetch_openml(name='creditcard',as_frame=True,parser='auto').frame

In [32]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 284807 entries, 0 to 284806
Data columns (total 30 columns):
 #   Column  Non-Null Count   Dtype   
---  ------  --------------   -----   
 0   V1      284807 non-null  float64 
 1   V2      284807 non-null  float64 
 2   V3      284807 non-null  float64 
 3   V4      284807 non-null  float64 
 4   V5      284807 non-null  float64 
 5   V6      284807 non-null  float64 
 6   V7      284807 non-null  float64 
 7   V8      284807 non-null  float64 
 8   V9      284807 non-null  float64 
 9   V10     284807 non-null  float64 
 10  V11     284807 non-null  float64 
 11  V12     284807 non-null  float64 
 12  V13     284807 non-null  float64 
 13  V14     284807 non-null  float64 
 14  V15     284807 non-null  float64 
 15  V16     284807 non-null  float64 
 16  V17     284807 non-null  float64 
 17  V18     284807 non-null  float64 
 18  V19     284807 non-null  float64 
 19  V20     284807 non-null  float64 
 20  V21     284807 non-null  f

In [33]:
import pandas as pd
y = pd.to_numeric(data.iloc[:,-1]).values
x = data.drop('Class', axis=1).to_numpy()
x.shape, y.shape

((284807, 29), (284807,))

In [34]:
x_train,x_test,y_train,y_test = train_test_split(x,y,stratify=y, test_size=0.2,random_state=42)
scaler = StandardScaler()
x_train = scaler.fit_transform(x_train)
x_test = scaler.transform(x_test)

x_train_t = torch.FloatTensor(x_train)
y_train_t = torch.FloatTensor(y_train)
x_test_t = torch.FloatTensor(x_test)
y_test_t = torch.FloatTensor(y_test)

In [35]:
class DetetFraud(nn.Module):
  def __init__(self, input_dim):
    super().__init__()
    self.network = nn.Sequential(
        nn.Linear(input_dim,64),
        nn.ReLU(),
        nn.Linear(64,32),
        nn.ReLU(),
        nn.Linear(32,1),        
    )
  def forward(self ,x):
    return self.network(x)    

# 불균형 비율 만큼 positive(1) 클래스에 가중치 부여  - 사기
num_pos = sum(y_train == 1)
num_neg = sum(y_train == 0)
pos_weight_val = torch.tensor([num_neg / num_pos])   # 정상데이터수 / 사기데이터수

model = DetetFraud(x_train_t.shape[-1])
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight_val)
optimizer  = Adam(model.parameters(), lr = 0.001)

In [36]:
from sklearn.metrics import classification_report
import torch

def evaluate_model(model, x, y):
  with torch.no_grad():
    logits = model(x)
    probs = torch.sigmoid(logits).numpy()
    preds = (probs >=0.5).astype(int)
  print(f'confusion matrix : {confusion_matrix(y, preds)}')
  print(classification_report(y,preds, target_names = ['0 정상', '1 사기']))
  return probs    

In [37]:
from torch.utils.data import TensorDataset, DataLoader
epochs = 20
batch_size = 2048
dataset = TensorDataset(x_train_t, y_train_t)
dataloader = DataLoader(dataset,batch_size=batch_size, shuffle=True)

from tqdm import tqdm
for epoch in tqdm(range(epochs)):
  total_loss = 0
  for batch_x, batch_y in dataloader:
    optimizer.zero_grad()
    output = model(batch_x).squeeze(1)  # (batch_size,1) -> (batch_size,)
    loss = criterion(output, batch_y)
    loss.backward()
    optimizer.step()
    total_loss += loss.item()
    
  avg_loss = total_loss / len(dataloader)    
  if (epoch+1) % 5 == 0:
    print(f'epoch : {epoch+1}/{epochs} 평균 loss = {avg_loss:.4f}')

 25%|██▌       | 5/20 [00:12<00:46,  3.08s/it]

epoch : 5/20 평균 loss = 0.2174


 50%|█████     | 10/20 [00:34<00:41,  4.14s/it]

epoch : 10/20 평균 loss = 0.1399


 75%|███████▌  | 15/20 [00:56<00:21,  4.36s/it]

epoch : 15/20 평균 loss = 0.0952


100%|██████████| 20/20 [01:18<00:00,  3.92s/it]

epoch : 20/20 평균 loss = 0.0675


In [38]:
test_probs = evaluate_model(model,x_test_t, y_test_t)

confusion matrix : [[55790  1074]
 [    8    90]]
              precision    recall  f1-score   support

        0 정상       1.00      0.98      0.99     56864
        1 사기       0.08      0.92      0.14        98

    accuracy                           0.98     56962
   macro avg       0.54      0.95      0.57     56962
weighted avg       1.00      0.98      0.99     56962

